In [0]:
from functools import reduce
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import IntegerType, TimestampType

In [0]:
tables: list = spark.catalog.listTables("bikes.01_bronze")
station_tables: list = [table.name for table in tables if "station_lookup_raw" in table.name]

dfs: list= []
for table in station_tables:
    df = spark.read.table(f"bikes.01_bronze.{table}")
    dfs.append(df)

merged_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)

In [0]:
merged_df = merged_df.select(
    col("city"),
    col("station_id"),
    col("name"),
    col("short_name"),
    col("lon").alias("longitude"),
    col("lat").alias("latitude"),
    col("region_id").cast(IntegerType()),
    col("capacity"),
    current_timestamp().alias("valid_from"),
    lit(None).cast(TimestampType()).alias('valid_to')
)

In [0]:
merged_df.write.mode("overwrite").saveAsTable("bikes.02_silver.stations_cleansed")